# =============================================================================
# PROJECT: COVID-19 Trend Analysis & Forecasting
# Author : Mahesh Maddileti
# Course : PGP in Data Science & Machine Learning — IntelliPaat (April 2026)
# Domain : Healthcare / Public Health
# =============================================================================

# =============================================================================
#
# PROBLEM STATEMENT:
# Given data about COVID-19 patients worldwide, visualize the impact of the pandemic, analyze infection and recovery trends, and forecast the number of cases expected in the next 7 days using Facebook Prophet.
#
# DATASET: Covid_19_Clean_Complete.csv
# Coverage: Global country-level daily data (up to July 2020)
# Columns: Date, Country/Region, Confirmed, Deaths, Recovered, Active
#
# KEY OBJECTIVES:
# 1. Visualize global COVID-19 spread using interactive choropleth maps
# 2. Identify top 10 countries by confirmed, deaths, active, and recovered
# 3. Compare India, US, and China trends over time
# 4. Build 7-day Prophet forecasting models for confirmed cases
# =============================================================================

In [ ]:
# ── SECTION 1: IMPORT LIBRARIES ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Update path if running locally
# For Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/Covid 19/Covid_19_Clean_Complete.csv')

df = pd.read_csv('Covid_19_Clean_Complete.csv')

In [ ]:
# Rename columns for easier access
df.rename(columns={
    'Province/State': 'state',
    'Country/Region': 'country'
}, inplace=True)

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(df.head())


In [ ]:
# ── SECTION 3: SNAPSHOT — LATEST DATE ANALYSIS ───────────────────────────────
# Get the most recent date in the dataset for current-state analysis
latest_date = df['Date'].max()
print(f"\nLatest date in dataset: {latest_date}")

In [ ]:
# Filter data for the latest date to get current standings
top = df[df['Date'] == latest_date]


In [ ]:
# Country-level summary for the latest date
country_summary = top.groupby(by='country')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().reset_index()
print(f"\nCountry summary on {latest_date}:")
print(country_summary.sort_values('Confirmed', ascending=False).head(10))


In [ ]:
# ── SECTION 4: GLOBAL TREND — CONFIRMED CASES OVER TIME ──────────────────────
# Aggregate confirmed cases across all countries by date
confirmed_global = df.groupby(by='Date')['Confirmed'].sum().reset_index()

In [ ]:
# Line plot showing global confirmed case growth
plt.figure(figsize=(15, 5))
sns.lineplot(data=confirmed_global, x='Date', y='Confirmed')
plt.title('Global COVID-19 Confirmed Cases Over Time', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Total Confirmed Cases')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()
# INSIGHT: Exponential growth pattern — casees accelerated rapidly from March 2020

In [ ]:
# 5.1 — Top 10 countries by Recovered cases
top_10_recovered = df.groupby(by='country')['Recovered'].sum().sort_values(ascending=False).head(10).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=top_10_recovered, x='country', y='Recovered', palette='Greens_r')
plt.title('Top 10 Countries — Most Recovered Cases', fontsize=14)
plt.xlabel('Country')
plt.ylabel('Total Recovered')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
# INSIGHT: US and Brazil dominate recovered counts due to high total case volume

In [ ]:
# 5.2 — Top 10 countries by Deaths
top_10_deaths = df.groupby(by='country')['Deaths'].sum().sort_values(ascending=False).head(10).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=top_10_deaths, x='country', y='Deaths', palette='Reds_r')
plt.title('Top 10 Countries — Most Deaths', fontsize=14)
plt.xlabel('Country')
plt.ylabel('Total Deaths')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
# INSIGHT: US has highest deaths, followed by Brazil and UK

In [ ]:
# 5.3 — Top 10 countries by Active cases
top_10_active = df.groupby(by='country')['Active'].sum().sort_values(ascending=False).head(10).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=top_10_active, x='country', y='Active', palette='Oranges_r')
plt.title('Top 10 Countries — Most Active Cases', fontsize=14)
plt.xlabel('Country')
plt.ylabel('Total Active Cases')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
 # ── SECTION 6: COUNTRY-LEVEL TIME SERIES COMPARISON ──────────────────────────
# Compare US, China, and India across Deaths, Recovered, and Confirmed cases

# Prepare country-specific time series data
us = df[df['country'] == 'US'].groupby('Date')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().reset_index()
china = df[df['country'] == 'China'].groupby('Date')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().reset_index()
india = df[df['country'] == 'India'].groupby('Date')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().reset_index()


In [ ]:
 # 6.1 — Deaths comparison: US vs China vs India
plt.figure(figsize=(15, 6))
sns.pointplot(x=us.index, y=us.Deaths, color='Red', label='US')
sns.pointplot(x=china.index, y=china.Deaths, color='Blue', label='China')
sns.pointplot(x=india.index, y=india.Deaths, color='Green', label='India')
plt.xlabel('No. of Days (from dataset start)', fontsize=12)
plt.ylabel('No. of Deaths', fontsize=12)
plt.title('Death Cases Comparison: US vs China vs India', fontsize=14)
plt.legend(['US', 'China', 'India'])
plt.tight_layout()
plt.show()
# INSIGHT: US deaths grew sharply from day ~60; China plateaued early; India slower rise


In [ ]:
# 6.2 — Recovered comparison
plt.figure(figsize=(15, 6))
sns.pointplot(x=us.index, y=us.Recovered, color='Red', label='US')
sns.pointplot(x=china.index, y=china.Recovered, color='Blue', label='China')
sns.pointplot(x=india.index, y=india.Recovered, color='Green', label='India')
plt.xlabel('No. of Days', fontsize=12)
plt.ylabel('No. of Recovered', fontsize=12)
plt.title('Recovery Comparison: US vs China vs India', fontsize=14)
plt.legend(['US', 'China', 'India'])
plt.tight_layout()
plt.show()
# INSIGHT: India's recovery curve accelerated sharply relative to its case count



In [ ]:
# 6.3 — Confirmed cases comparison
plt.figure(figsize=(15, 6))
sns.pointplot(x=us.index, y=us.Confirmed, color='Red', label='US')
sns.pointplot(x=china.index, y=china.Confirmed, color='Blue', label='China')
sns.pointplot(x=india.index, y=india.Confirmed, color='Green', label='India')
plt.xlabel('No. of Days', fontsize=12)
plt.ylabel('No. of Confirmed Cases', fontsize=12)
plt.title('Confirmed Cases Comparison: US vs China vs India', fontsize=14)
plt.legend(['US', 'China', 'India'])
plt.tight_layout()
plt.show()

In [ ]:
# ── SECTION 7: TIME-SERIES FORECASTING WITH FACEBOOK PROPHET ─────────────────
# Prophet is an open-source forecasting library by Facebook/Meta
# It works best with daily time-series data and handles seasonality automatically
# Prophet requires columns named 'ds' (date) and 'y' (target value)

# Install if not already: pip install prophet
from prophet import Prophet

# Reload with proper datetime parsing
df2 = pd.read_csv('Covid_19_Clean_Complete.csv', parse_dates=['Date'])

# Prepare global confirmed cases time series
confirmed_ts = df2.groupby(by='Date').sum()['Confirmed'].reset_index()
confirmed_ts.columns = ['ds', 'y']  # Prophet requires these column names

print("\nPreparing Prophet model for Confirmed Cases...")

# Initialize and fit the Prophet model
m = Prophet(
    interval_width=0.95,     # 95% confidence interval
    yearly_seasonality=True  # Enable yearly seasonality detection
)
m.fit(confirmed_ts)

# Create future dataframe for 7-day forecast
future = m.make_future_dataframe(periods=7)
forecast = m.predict(future)

# Display forecast results
print("\n7-Day Forecast — Confirmed Cases:")
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10))

# Plot forecast
m.plot(forecast)
plt.title('COVID-19 Confirmed Cases — 7-Day Forecast', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Confirmed Cases')
plt.tight_layout()
plt.show()
# INSIGHT: yhat = predicted value | yhat_lower/upper = confidence interval bounds


In [ ]:
 # ── SECTION 8: INTERACTIVE CHOROPLETH MAP (PLOTLY) ───────────────────────────
# Interactive world map showing active COVID-19 cases by country
# Darker red = more active cases

import plotly.express as px

world = df2.groupby(by='Country/Region')[['Confirmed', 'Active', 'Deaths', 'Recovered']].sum().reset_index()

figure = px.choropleth(
    world,
    locations='Country/Region',
    locationmode='country names',
    color='Active',
    hover_name='Country/Region',
    range_color=[1, 20000],
    color_continuous_scale='Reds',
    title='Global Active COVID-19 Cases by Country'
)
figure.show()
# INSIGHT: US, Brazil, and India have the darkest shading — highest active case counts


In [ ]:
# ── SECTION 9: KEY FINDINGS SUMMARY ──────────────────────────────────────────
print("\n" + "="*60)
print("KEY FINDINGS")
print("="*60)
print("1. US had the highest confirmed cases and deaths globally")
print("2. China showed early flattening — effective early containment")
print("3. India's recovery rate accelerated faster than US/Brazil")
print("4. Prophet forecast shows continued case growth in 7-day window")
print("5. Holiday/seasonal patterns visible in trend decomposition")
print("6. Top 10 deaths: US, Brazil, UK, Mexico, Italy, France,")
print("   Spain, India, Iran, Peru")
